# 실습 문제 모범답안 - 320. Handoffs Pattern

`320_Handoffs_Pattern` 노트북 실습 문제에 대한 모범답안 예시입니다.

### 구현 요약

**보증 기간 내 + 소프트웨어 문제 시나리오**

1. 새 `thread_id`로 새로운 대화 세션 시작
2. 4턴 멀티 턴 대화: 문제 호소 → 보증 기간 내 응답 → 소프트웨어 문제 설명 → 해결책 요청
3. 각 턴마다 `current_step`, `warranty_status`, `issue_type` 상태 전환 확인

**기대 결과**: `warranty_status="in_warranty"`, `issue_type="software"` 가 기록되고,
마지막 단계에서 `escalate_to_human` 이 아니라 `provide_solution` 도구가 호출됩니다.
(에스컬레이션은 "보증 만료 + 하드웨어" 경우에만 발생)

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gpt-5-mini", model_provider="openai")

## 사전 준비: 본문 노트북의 구성 요소 재정의

이 노트북을 단독으로 실행할 수 있도록, 본문(320_Handoffs_Pattern)에서 만든
상태 스키마, 상태 관리 도구, 단계별 프롬프트, 미들웨어, 에이전트를 다시 정의합니다.

In [3]:
from typing import Literal, Callable
from typing_extensions import NotRequired
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver


# --- 상태 스키마 (본문 1장) ---
class SupportState(AgentState):
    """고객 지원 워크플로우 상태"""
    current_step: NotRequired[Literal["warranty_collector", "issue_classifier", "resolution_specialist"]]
    warranty_status: NotRequired[Literal["in_warranty", "out_of_warranty"]]
    issue_type: NotRequired[Literal["hardware", "software"]]


# --- 상태 관리 도구 (본문 2장) ---
@tool
def record_warranty_status(
    status: Literal["in_warranty", "out_of_warranty"],
    runtime: ToolRuntime[None, SupportState],
) -> Command:
    """고객의 보증 상태를 기록하고 이슈 분류 단계로 전환합니다."""
    return Command(update={
        "messages": [ToolMessage(
            content=f"Warranty status recorded as: {status}",
            tool_call_id=runtime.tool_call_id,
        )],
        "warranty_status": status,
        "current_step": "issue_classifier",
    })

@tool
def record_issue_type(
    issue_type: Literal["hardware", "software"],
    runtime: ToolRuntime[None, SupportState],
) -> Command:
    """문제 유형을 기록하고 해결 전문가 단계로 전환합니다."""
    return Command(update={
        "messages": [ToolMessage(
            content=f"Issue type recorded as: {issue_type}",
            tool_call_id=runtime.tool_call_id,
        )],
        "issue_type": issue_type,
        "current_step": "resolution_specialist",
    })

@tool
def escalate_to_human(reason: str) -> str:
    """고객 문제를 사람 상담원에게 에스컬레이션합니다."""
    return f"Escalating to human support. Reason: {reason}"

@tool
def provide_solution(solution: str) -> str:
    """고객에게 문제 해결 방안을 제공합니다."""
    return f"Solution provided: {solution}"


# --- 단계별 프롬프트 (본문 3장) ---
WARRANTY_COLLECTOR_PROMPT = """당신은 고객 지원 에이전트입니다.

현재 단계: 보증 확인

이 단계에서 해야 할 일:
1. 고객에게 친절하게 인사하세요
2. 기기가 보증 기간 내인지 물어보세요
3. record_warranty_status 도구를 사용하여 응답을 기록하고 다음 단계로 이동하세요

대화적이고 친근하게 대화하세요. 한 번에 여러 질문을 하지 마세요."""

ISSUE_CLASSIFIER_PROMPT = """당신은 고객 지원 에이전트입니다.

현재 단계: 이슈 분류
고객 정보: 보증 상태는 {warranty_status}입니다

이 단계에서 해야 할 일:
1. 고객에게 문제를 설명해달라고 요청하세요
2. 하드웨어 문제(물리적 손상, 부품 고장)인지 소프트웨어 문제(앱 충돌, 성능 저하)인지 판단하세요
3. record_issue_type 도구를 사용하여 분류를 기록하고 다음 단계로 이동하세요

불명확한 경우 분류하기 전에 명확히 질문하세요."""

RESOLUTION_SPECIALIST_PROMPT = """당신은 고객 지원 에이전트입니다.

현재 단계: 해결책 제공
고객 정보: 보증 상태는 {warranty_status}, 이슈 유형은 {issue_type}입니다

이 단계에서 해야 할 일:
1. 소프트웨어 문제인 경우: provide_solution을 사용하여 문제 해결 단계를 제공하세요
2. 하드웨어 문제인 경우:
   - 보증 기간 내: provide_solution을 사용하여 보증 수리 프로세스를 설명하세요
   - 보증 기간 만료: escalate_to_human을 사용하여 유료 수리 옵션을 위해 인간 지원으로 에스컬레이션하세요

구체적이고 도움이 되는 해결책을 제공하세요."""


# --- 단계별 설정 및 미들웨어 (본문 4장) ---
STEP_CONFIG = {
    "warranty_collector": {
        "prompt": WARRANTY_COLLECTOR_PROMPT,
        "tools": [record_warranty_status],
        "requires": [],
    },
    "issue_classifier": {
        "prompt": ISSUE_CLASSIFIER_PROMPT,
        "tools": [record_issue_type],
        "requires": ["warranty_status"],
    },
    "resolution_specialist": {
        "prompt": RESOLUTION_SPECIALIST_PROMPT,
        "tools": [provide_solution, escalate_to_human],
        "requires": ["warranty_status", "issue_type"],
    },
}

@wrap_model_call
def apply_step_config(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    """현재 단계에 맞는 시스템 프롬프트와 도구를 동적으로 적용하는 미들웨어"""
    current_step = request.state.get("current_step", "warranty_collector")
    stage_config = STEP_CONFIG[current_step]

    for key in stage_config["requires"]:
        if request.state.get(key) is None:
            raise ValueError(f"'{current_step}' 단계 진입에 필요한 상태가 없습니다: {key}")

    system_prompt = stage_config["prompt"].format(
        warranty_status=request.state.get("warranty_status", "알 수 없음"),
        issue_type=request.state.get("issue_type", "알 수 없음"),
    )

    request = request.override(
        system_prompt=system_prompt,
        tools=stage_config["tools"],
    )
    return handler(request)


# --- 에이전트 생성 (본문 5장) ---
agent = create_agent(
    model,
    tools=[record_warranty_status, record_issue_type, provide_solution, escalate_to_human],
    state_schema=SupportState,
    middleware=[apply_step_config],
    checkpointer=InMemorySaver(),
)

## 시나리오 실행: 보증 기간 내 + 소프트웨어 문제

새 스레드에서 4턴 대화를 진행합니다.

- 턴 1~3: `invoke`로 실행하고 턴이 끝날 때마다 상태 전환만 확인
- 턴 4: 본문처럼 `stream_mode="values"`로 실행해 이번 턴에 새로 생성되는 메시지만 출력
  (`invoke`의 `result["messages"]`는 전체 대화 이력이라 출력이 너무 길어짐)

In [4]:
import uuid
from langchain_core.messages import HumanMessage

# 새로운 대화 세션 시작
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

# 턴 1 ~ 턴 3: 상태 전환만 확인
turns = [
    "안녕하세요, 휴대폰 앱이 자꾸 강제 종료돼요.",                                # 턴 1: 문제 호소
    "네, 아직 보증 기간 내예요.",                                              # 턴 2: 보증 기간 내
    "떨어뜨린 적은 없어요. 최근 업데이트 후부터 앱이 계속 꺼지는 소프트웨어 문제 같아요.",  # 턴 3: 소프트웨어 문제
]

for i, user_input in enumerate(turns, 1):
    print("\n" + "=" * 80)
    print(f"턴 {i}: {user_input}")
    print("=" * 80)

    result = agent.invoke(
        {"messages": [HumanMessage(content=user_input)]},
        config=config,
    )

    # 턴이 끝날 때마다 상태 전환 확인
    print(f"현재 단계: {result.get('current_step', 'N/A')}")
    print(f"보증 상태: {result.get('warranty_status', 'N/A')}")
    print(f"이슈 유형: {result.get('issue_type', 'N/A')}")


턴 1: 안녕하세요, 휴대폰 앱이 자꾸 강제 종료돼요.
현재 단계: N/A
보증 상태: N/A
이슈 유형: N/A

턴 2: 네, 아직 보증 기간 내예요.
현재 단계: resolution_specialist
보증 상태: in_warranty
이슈 유형: software

턴 3: 떨어뜨린 적은 없어요. 최근 업데이트 후부터 앱이 계속 꺼지는 소프트웨어 문제 같아요.
현재 단계: resolution_specialist
보증 상태: in_warranty
이슈 유형: software


In [5]:
# 턴 4: 해결책 요청 — stream_mode="values" 로 이번 턴에 새로 생성되는 메시지만 출력
# (invoke 의 result["messages"] 는 전체 대화 이력이라 출력이 길어짐)
print("=" * 80)
print("턴 4: 어떻게 해결하면 되나요? (provide_solution 호출 확인)")
print("=" * 80)

for step in agent.stream(
    {"messages": [HumanMessage(content="어떻게 해결하면 되나요?")]},
    config=config,
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

# 최종 상태 확인
print(f"\n현재 단계: {step.get('current_step', 'N/A')}")
print(f"보증 상태: {step.get('warranty_status', 'N/A')}")
print(f"이슈 유형: {step.get('issue_type', 'N/A')}")

턴 4: 어떻게 해결하면 되나요? (provide_solution 호출 확인)
================================ Human Message =================================

어떻게 해결하면 되나요?
================================== Ai Message ==================================
Tool Calls:
  provide_solution (call_T0HKlTRDsBoxrTbxiC8nlurn)
 Call ID: call_T0HKlTRDsBoxrTbxiC8nlurn
  Args:
    solution: 아래 단계대로 차례로 진행해 보세요. 문제가 ‘최근 업데이트 후’ 시작됐다면 재설치·캐시 초기화와 로그 수집이 가장 빠르게 원인을 찾습니다. 진행 중 막히면 어느 단계에서 막혔는지 알려주시면 단계별로 원격으로 안내해 드리겠습니다.

1) 기기 재시작
- 전원 끄기 → 30초 대기 → 켜기. 간단하지만 먼저 해보세요.

2) 앱·OS 최신화 확인
- 앱스토어(Play Store / App Store)에서 앱이 최신 버전인지 확인 후 업데이트(또는 업데이트가 문제 시작 시점이면 다음 단계로).
- 설정 → 일반/시스템 → 소프트웨어 업데이트에서 OS 최신화 여부 확인.

3) 앱 캐시/데이터 초기화 또는 앱 삭제·재설치(가장 효과적)
- Android: 설정 → 앱 → 해당 앱 → 저장공간 → 캐시 삭제. 해결 안 되면 ‘데이터 삭제’(로그인 정보/설정 초기화) 또는 앱 삭제 후 재설치.
- iOS: 앱 자체 캐시 삭제 불가 → 앱 삭제 → 기기 재부팅 → App Store에서 재설치. 재설치 전 필요한 데이터(백업)가 있는지 확인.

4) 권한·배터리 최적화 확인
- 설정 → 앱 → 해당 앱 → 권한에서 필요한 권한 허용(저장공간, 카메라 등).
- Android: 설정 → 배터리 → 배터리 최적화에서 해당 앱을 예외로 추가.
- iOS: 저전력모드가 켜

## 확인 포인트

- 턴 2 이후: `warranty_status = "in_warranty"`, 현재 단계가 `issue_classifier`로 전환
- 턴 3 이후: `issue_type = "software"`, 현재 단계가 `resolution_specialist`로 전환
- 턴 4: 소프트웨어 문제이므로 보증 상태와 관계없이 **`provide_solution`** 이 호출되어 해결 단계가 안내됩니다.
  `escalate_to_human`은 "보증 만료 + 하드웨어" 조합에서만 호출됩니다 (본문 8장 시나리오와 비교해 보세요).